# 01 — SparkSession & DataFrame Basics

**What you will learn:**
- What SparkSession is and how to access it in Databricks
- Creating DataFrames from Python lists and explicit schemas
- Inspecting a DataFrame (schema, columns, dtypes, shape)
- Basic actions: `show()`, `count()`, `first()`, `head()`, `take()`
- Converting between Spark DataFrame and Pandas DataFrame

## 1. SparkSession

In Databricks, SparkSession is **already created** for you as `spark`.
You never need to call `SparkSession.builder...` manually.
It is the single entry point to all Spark functionality.

In [ ]:
# Confirm SparkSession is available
print(spark)
print("Spark version:", spark.version)
print("App name     :", spark.sparkContext.appName)

## 2. Creating a DataFrame from a Python List

In [ ]:
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType, DateType, BooleanType
)

# --- 2a. Simplest way: infer schema from list of tuples ---
data = [
    (1, "Alice",   "Engineering", 95000.0, "2022-01-15"),
    (2, "Bob",     "Marketing",   72000.0, "2021-06-01"),
    (3, "Charlie", "Engineering", 105000.0,"2020-03-20"),
    (4, "Diana",   "HR",          68000.0, "2023-09-10"),
    (5, "Eve",     "Marketing",   78000.0, "2022-11-05"),
]

columns = ["emp_id", "name", "department", "salary", "hire_date"]

df = spark.createDataFrame(data, schema=columns)
df.show()

## 3. Creating a DataFrame with an Explicit Schema

Always define a schema explicitly for production code — it is faster
(no schema inference scan) and avoids type surprises.

In [ ]:
schema = StructType([
    StructField("emp_id",     IntegerType(), nullable=False),
    StructField("name",       StringType(),  nullable=False),
    StructField("department", StringType(),  nullable=True),
    StructField("salary",     DoubleType(),  nullable=True),
    StructField("hire_date",  StringType(),  nullable=True),
])

df_typed = spark.createDataFrame(data, schema=schema)
df_typed.printSchema()
df_typed.show()

## 4. Inspecting a DataFrame

In [ ]:
# Column names
print("Columns :", df_typed.columns)

# Data types as list of (name, type_string) tuples
print("DTypes  :", df_typed.dtypes)

# Number of rows
print("Row count:", df_typed.count())

# Number of columns
print("Col count:", len(df_typed.columns))

In [ ]:
# printSchema — tree view of the schema
df_typed.printSchema()

## 5. show(), first(), head(), take()

In [ ]:
# show(n) — prints first n rows (default 20), truncates long strings
df_typed.show(3)

# show with no truncation
df_typed.show(truncate=False)

In [ ]:
# first() — returns the first Row object (driver action)
first_row = df_typed.first()
print(type(first_row))   # pyspark.sql.types.Row
print(first_row)
print("Name:", first_row["name"])
print("Name:", first_row.name)       # attribute-style access

In [ ]:
# head(n) — returns list of first n Row objects
rows = df_typed.head(3)
for r in rows:
    print(r)

In [ ]:
# take(n) — same as head(n), returns list of Row
rows = df_typed.take(2)
print(rows)

## 6. Summary Statistics

In [ ]:
# describe() — count, mean, stddev, min, max for numeric/string cols
df_typed.describe("salary").show()

# summary() — adds percentile information
df_typed.summary().show()

## 7. Converting to Pandas DataFrame

`toPandas()` collects ALL data to the driver — use only on small datasets.

In [ ]:
pandas_df = df_typed.toPandas()
print(type(pandas_df))
print(pandas_df)

In [ ]:
# Convert Pandas back to Spark DataFrame
import pandas as pd

pdf = pd.DataFrame({
    "city":  ["New York", "London", "Tokyo"],
    "code":  ["NYC", "LDN", "TKY"],
    "pop_m": [8.3, 9.0, 13.9]
})

spark_df = spark.createDataFrame(pdf)
spark_df.show()

## 8. Creating DataFrames with Row Objects

In [ ]:
rows = [
    Row(product="Laptop",  category="Electronics", price=999.99),
    Row(product="Desk",    category="Furniture",   price=349.00),
    Row(product="Monitor", category="Electronics", price=499.00),
]

df_products = spark.createDataFrame(rows)
df_products.printSchema()
df_products.show()

## 9. Spark DataFrame vs RDD

| | DataFrame | RDD |
|---|---|---|
| Abstraction | Structured (rows + schema) | Low-level distributed collection |
| Optimization | Catalyst optimizer (automatic) | Manual |
| API style | SQL-like, column expressions | Functional (map/filter/reduce) |
| Preferred for | All structured data tasks | Custom serialization, legacy code |

Access the underlying RDD if needed:

In [ ]:
rdd = df_typed.rdd
print(type(rdd))
print(rdd.take(2))

## 10. Caching a DataFrame

Cache a DataFrame when you reuse it multiple times to avoid recomputation.

In [ ]:
df_typed.cache()          # lazy — actual caching happens on first action
df_typed.count()          # triggers caching

# Check if cached
print("Is cached:", df_typed.is_cached)

# Release cache when done
df_typed.unpersist()
print("Is cached after unpersist:", df_typed.is_cached)

## Key Takeaways

| Concept | Notes |
|---|---|
| `spark` | Pre-created SparkSession in every Databricks notebook |
| `createDataFrame(data, schema)` | Preferred way — always pass explicit schema |
| `printSchema()` | Tree view of column names and types |
| `show(n, truncate)` | Displays data; never use for large datasets |
| `first()` / `take(n)` | Fetch rows to driver — keep n small |
| `toPandas()` | Collects all data to driver — only for small DataFrames |
| `cache()` / `unpersist()` | Cache reused DataFrames; always release when done |